In [1]:
import os
import pandas as pd

import openai
from dotenv import load_dotenv, find_dotenv

In [2]:
from openai.resources.evals.graders import PointWiseStringCheckGrader, DirectCheckOp
from openai.resources.evals.pipeline import EvalPipeline, ReturnFormat
from openai.resources.evals.data_sources import DataSource

In [3]:
print('openai version:', openai.__version__)

openai version: 1.82.0-dev-evals


In [4]:
load_dotenv(find_dotenv(), override=True)
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
if not OPENAI_API_KEY:
    raise EnvironmentError("OPENAI_API_KEY not set. Please add it to your .env file.")

In [5]:
client = openai.OpenAI()

In [6]:
ground_truth = [
    "foo",
    "bar",
    "qux",
]
predicted = [
    "foo",
    "bar",
    "baz",
]

eval_dataset = pd.DataFrame({
    "predicted" : predicted,
    "ground_truth" : ground_truth,
})

In [7]:
eval_dataset.head()

,predicted,ground_truth
0,foo,foo
1,bar,bar
2,baz,qux


In [8]:
string_checker_point_wise = PointWiseStringCheckGrader(
    input="{{ item.ground_truth }}",
    reference="{{ item.predicted }}",
    operation=DirectCheckOp.EQUALS,
)

ds = DataSource.from_dataframe(eval_dataset)

In [9]:
ds.as_tuple()

({'type': 'custom',
  'item_schema': {'type': 'object',
   'properties': {'predicted': {'type': 'string'},
    'ground_truth': {'type': 'string'}},
   'required': ['predicted', 'ground_truth']}},
 {'type': 'jsonl',
  'source': {'type': 'file_content',
   'content': [{'item': {'predicted': 'foo', 'ground_truth': 'foo'}},
    {'item': {'predicted': 'bar', 'ground_truth': 'bar'}},
    {'item': {'predicted': 'baz', 'ground_truth': 'qux'}}]}})

In [10]:
eval_task = EvalPipeline(
    data_source=ds,  
    graders=[string_checker_point_wise],
)

In [11]:

result = eval_task.run_pipeline(return_format=ReturnFormat.DATAFRAME) # type: ignore

Fetched items: 3
All items: [{'ground_truth': 'qux', 'predicted': 'baz'}, {'ground_truth': 'bar', 'predicted': 'bar'}, {'ground_truth': 'foo', 'predicted': 'foo'}]


In [12]:
result.head() # type: ignore

,id,created_at,datasource_item,datasource_item_id,eval_id,object,results,run_id,status,datasource_item.ground_truth,datasource_item.predicted,result.name,result.sample,result.passed,result.score
0,outputitem_68399613aafc8191a9b3214803f84344,1748604435,"{'ground_truth': 'qux', 'predicted': 'baz'}",2,eval_6839960d9cc08191bf5912b02085d91f,eval.run.output_item,[{'name': 'PointWiseStringCheckGrader-2ce23fb3...,evalrun_6839960dfc308191a019d938ad47677c,fail,qux,baz,PointWiseStringCheckGrader-2ce23fb3-b5bc-4583-...,None,False,0.0
1,outputitem_683996139730819180c53360908b0bc7,1748604435,"{'ground_truth': 'bar', 'predicted': 'bar'}",1,eval_6839960d9cc08191bf5912b02085d91f,eval.run.output_item,[{'name': 'PointWiseStringCheckGrader-2ce23fb3...,evalrun_6839960dfc308191a019d938ad47677c,pass,bar,bar,PointWiseStringCheckGrader-2ce23fb3-b5bc-4583-...,None,True,1.0
2,outputitem_6839961382948191a13df027209af4e2,1748604435,"{'ground_truth': 'foo', 'predicted': 'foo'}",0,eval_6839960d9cc08191bf5912b02085d91f,eval.run.output_item,[{'name': 'PointWiseStringCheckGrader-2ce23fb3...,evalrun_6839960dfc308191a019d938ad47677c,pass,foo,foo,PointWiseStringCheckGrader-2ce23fb3-b5bc-4583-...,None,True,1.0
